# AI Smart Agriculture: Model Fine-Tuning Notebook

Welcome! This notebook will guide you through the process of fine-tuning our MobileNetV2 model to improve its accuracy on the PlantVillage dataset. 

**Instructions:**
1.  Make sure you have uploaded the `PlantVillage.zip` file to your Google Drive.
2.  Run each code cell in order by clicking the "Play" button on the left side of the cell.
3.  Read the markdown text cells (like this one) for instructions.

## Step 1: Connect to Google Drive

This first cell will connect the Colab environment to your Google Drive. You will be asked to click a link to authorize the connection and then paste an authorization code back into the input box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Unzip the Dataset

This cell will unzip your dataset from Google Drive into the Colab environment. This is much faster than accessing individual image files from Drive directly.

**! ! ! CRITICAL ! ! !**
You **MUST** check that the path in the command below matches the location where you saved the `PlantVillage.zip` file in your Google Drive. If you created a `Colab_Datasets` folder as suggested, this path should be correct. If not, please edit the path accordingly.

In [ ]:
# Verify this path is correct for your Google Drive!
!unzip /content/drive/MyDrive/Colab_Datasets/PlantVillage.zip -d /content/

## Step 3: Imports and Configuration

Here, we import the necessary libraries and set up our main configuration variables.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

# --- Configuration ---
# The unzipped data is now in the /content/PlantVillage folder
DATA_DIR = '/content/PlantVillage' 
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 38

## Step 4: Data Preprocessing and Augmentation

We set up the `ImageDataGenerator` to normalize our image data and create augmented versions (rotated, zoomed, flipped) to help the model generalize better and prevent overfitting.

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values to 0-1
    validation_split=0.2,    # Reserve 20% of data for validation
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

## Step 5: Define the Model Architecture for Fine-Tuning

This is the most important part. We will:
1. Load the pre-trained MobileNetV2 model.
2. **Freeze** the entire base model initially.
3. **Unfreeze** the top layers of the base model, allowing them to be trained.
4. Add our custom classification head on top.
5. Compile the final model with a very **low learning rate**, which is critical for fine-tuning.

In [ ]:
# 1. Load the base model
base_model = MobileNetV2(
    weights='imagenet', 
    include_top=False,  # Exclude the final classification layer
    input_shape=(224, 224, 3)
)

# 2. Unfreeze the top layers for fine-tuning
base_model.trainable = True

# Let's freeze the first 100 layers and fine-tune the rest
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Add our custom classification head
x = base_model.output
x = Flatten()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# 4. Compile with a low learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5), # CRITICAL for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 6: Train the Model

Now we run the training process. We will use `ModelCheckpoint` to automatically save the weights of the epoch that has the best validation accuracy. 

This step will take **approximately 20-30 minutes** to complete with a GPU.

In [ ]:
EPOCHS = 10

# Define a new name for our improved weights file
checkpoint_filepath = 'smart_agri_weights_v2.weights.h5'

model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True, # We only need the weights
    monitor='val_accuracy', # Monitor validation accuracy
    mode='max',             # Save the model with the highest val_accuracy
    save_best_only=True)    # Only save if it's the best so far

print("Starting fine-tuning...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=[model_checkpoint_callback] # Add our callback here
)
print("Fine-tuning complete!")

## Step 7: Download Your New Model Weights

Congratulations! Your model has been fine-tuned. The best version of the weights has been saved to the file `smart_agri_weights_v2.weights.h5`.

**Action:**
1.  Click the **folder icon** in the left sidebar.
2.  You will see the file `smart_agri_weights_v2.weights.h5` in the file list.
3.  Right-click on the file and choose **Download**.
4.  Save it to your local computer in your `Smart_Agri_System` project folder.

You can now use this new weights file in your Flask application to get more accurate predictions.